Рекомендую запускать данный код в `google colab`, там этот ноутбук был протестирован. Но если решили запускать локально, то создайте отдельное окружение, так как можно не попасть с версиями библиотек.

Работаем с небольшой выборкой русскоязычных новостей. При первом запуске загружаются исходные статьи и сохраняются в папку `news_cache`. Затем можно менять темы и число статей без повторной загрузки.

Для работы нужны `spacy`, `pandas`, `requests`, `scikit-learn` и модель `ru_core_news_sm`. Установите зависимости в том же окружении, в котором запущен Jupyter.

In [1]:
%pip install -q spacy pandas scikit-learn datasets ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.8.0/ru_core_news_sm-3.8.0-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
from collections import Counter
import pandas as pd
import spacy
from datasets import load_dataset, load_from_disk

DATASET = 'data-silence/rus_news_classifier'
DATA_PATH = Path('news_dataset')
TOPICS = {3: 'economy', 7: 'science', 9: 'sports'} # это можно потрогать, что бы данные были более сложные
COUNT_EXP = 45 # если данных вам мало, то можно увеличить

if DATA_PATH.exists():
    dataset = load_from_disk(str(DATA_PATH))
else:
    dataset = load_dataset(DATASET)
    dataset.save_to_disk(str(DATA_PATH))

def select_articles(split):
    frame = dataset[split].to_pandas()[['news', 'labels']]
    frame = frame.rename(columns={'news': 'text', 'labels': 'label'})
    frame = frame[frame['label'].isin(TOPICS)].copy()
    frame['text'] = frame['text'].fillna('').astype(str).str.strip()
    frame = frame[frame['text'].str.split().str.len() >= 35]
    frame['topic'] = frame['label'].map(TOPICS)
    frame['source_split'] = split
    frame = frame.drop_duplicates(subset='text')
    return frame.groupby('label', group_keys=False).head(COUNT_EXP)

texts_df = pd.concat([select_articles('train'), select_articles('test')], ignore_index=True)
print('Total:', len(texts_df))
print(pd.crosstab(texts_df.source_split, texts_df.topic))
display(texts_df[['topic', 'text']].assign(text=lambda x: x.text.str.slice(0, 130)).head())

c:\Users\Varvara\MIPT\mipt2026f-nlp-karyakina-v-a\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (1/1 shards): 100%|██████████| 14383/14383 [00:00<00:00, 157758.04 examples/s]


Total: 270
topic         economy  science  sports
source_split                          
test               45       45      45
train              45       45      45


,topic,text
0,science,В 2021 году Microsoft выпустит специальную вер...
1,sports,Британский боксер-тяжеловес Тайсон Фьюри оскор...
2,science,Нидерландская компания U-Boat Worx представила...
3,science,Компания Xiaomi до конца осени может начать оф...
4,economy,Для учеников 8–11-х классов открыта регистраци...


## 1. Подготовка текстов

Для обязательной части задания возьмем 12 статей: по четыре на каждую тему. Обработка выполняется с помощью `ru_core_news_sm`.

In [4]:
try:
    nlp = spacy.load('ru_core_news_sm')
except OSError as exc:
    raise RuntimeError('Установите ru_core_news_sm с помощью предыдущей ячейки') from exc

selected = (texts_df[texts_df.source_split == 'train'].groupby('topic', group_keys=False).head(4).reset_index(drop=True).copy())
docs = list(nlp.pipe(selected.text.tolist(), batch_size=8))
print('Темы:', selected.topic.value_counts().to_dict())
print('Пример:', docs[0].text[:240])

Темы: {'science': 4, 'sports': 4, 'economy': 4}
Пример: В 2021 году Microsoft выпустит специальную версию Windows для бюджетных компьютеров. Об этом сообщает издание Windows Latest. Журналисты выяснили, что операционная система для ПК с двумя экранами — Windows 10X — будет иметь особую версию, р


## 2. Морфологический анализ

Напишите `morph_profile(doc)`. Функция должна возвращать словарь с полями:

- `words` - количество слов;
- `sentences` - количество предложений;
- `nouns`, `verbs`, `adjectives` - количество слов с соответствующими POS;
- `top_lemmas` - пять самых частых лемм существительных, глаголов и прилагательных, без стопслов.

Выведите результаты для двух статей разных тем.


In [27]:
def morph_profile(doc):
    # Your code here
    nouns = [t for t in doc if t.pos_ == "NOUN"]
    verbs = [t for t in doc if t.pos_ == "VERB"]
    adjs = [t for t in doc if t.pos_ == "ADJ"]

    def top_lemmas(tokens, n=5):
        counter = Counter(
            t.lemma_.lower()
            for t in tokens
            if t.is_alpha and not t.is_stop and t.lemma_.strip()
        )
        return counter.most_common(n)
    
    return {
            "words": sum(t.is_alpha for t in doc), 
            "sentences": sum(1 for _ in doc.sents), 
            "nouns": len(nouns), 
            "verbs": len(verbs), 
            "adjectives": len(adjs),
            "top_lemmas": {
                "nouns": top_lemmas(nouns),
                "verbs": top_lemmas(verbs),
                "adjectives":  top_lemmas(adjs),
        },
    }

# Your code here
news = selected.groupby('topic', group_keys=False).head(1).head(2)

for idx, row in news.iterrows():
    doc = docs[idx]         
    p   = morph_profile(doc)

    print(f"Topic: {row['topic']} (doc {idx})")
    print(f"Text: {row['text'][:160]}…")
    print(f"Words: {p['words']} | Sentences: {p['sentences']}")
    print(f"NOUN: {p['nouns']} | VERB: {p['verbs']} | ADJ: {p['adjectives']}")
    for cat in ("nouns", "verbs", "adjectives"):
        tops = ", ".join(f"{w} ({c})" for w, c in p["top_lemmas"][cat]) or "—"
        print(f"Top-5 {cat}: {tops}")
    print()

Topic: science (doc 0)
Text: В 2021 году Microsoft выпустит специальную версию Windows для бюджетных компьютеров. Об этом сообщает издание Windows Latest. Журналисты выяснили, что операцион…
Words: 224 | Sentences: 16
NOUN: 70 | VERB: 32 | ADJ: 28
Top-5 nouns: система (7), год (5), компьютер (5), версия (4), экран (4)
Top-5 verbs: появиться (3), выпустить (2), рассчитать (2), сообщаться (2), сообщать (1)
Top-5 adjectives: операционный (5), рабочий (2), специальный (1), бюджетный (1), слабый (1)

Topic: sports (doc 1)
Text: Британский боксер-тяжеловес Тайсон Фьюри оскорбил украинца Александра Усика. Его слова приводит Daily Mail. Спортсмен назвал возможного соперника маленьким укра…
Words: 126 | Sentences: 13
NOUN: 35 | VERB: 15 | ADJ: 20
Top-5 nouns: пояс (3), украинец (2), бой (2), победа (2), организация (2)
Top-5 verbs: заявить (2), оскорбить (1), приводить (1), назвать (1), являться (1)
Top-5 adjectives: боксёрский (4), чемпионский (2), всемирный (2), международный (2), британский 

## 3. Синтаксические связи

Реализуйте `verb_relations(doc)`: для каждого токена с POS `VERB` найдите непосредственно зависимые от него подлежащее (`nsubj`) и дополнение (`obj`). Возвращайте список словарей с ключами `verb`, `subject`, `object`. Значения - леммы в нижнем регистре. Если зависимого слова нет, используйте `None`. Если подходящих слов несколько, достаточно первого.

Пример формата результата: `{"verb": "проверить", "subject": "инженер", "object": "двигатель"}`.

Запустите функцию на трех статьях разных тем. Найдите один случай, когда такое правило пропускает важную связь или дает неточный результат.


In [28]:
def verb_relations(doc):
    relations = []
    # Your code here

    for token in doc:
        if token.pos_ != "VERB":
            continue

        subj = next((c for c in token.children if c.dep_ == "nsubj"), None)
        obj  = next((c for c in token.children if c.dep_ == "obj"),   None)

        relations.append({
            "verb":    token.lemma_.lower(),
            "subject": subj.lemma_.lower() if subj is not None else None,
            "object":  obj.lemma_.lower()  if obj  is not None else None,
        })
    return relations

# Your code here
news = selected.groupby('topic', group_keys=False).head(1).head(3)

for idx, row in news.iterrows():
    doc = docs[idx]             
    print(f"Topic: {row['topic']} (doc {idx})")
    print(f"Text: {row['text'][:160].replace(chr(10), ' ')}…")

    relations = verb_relations(doc)

    for rel in relations[:5]:
        print(rel)

Topic: science (doc 0)
Text: В 2021 году Microsoft выпустит специальную версию Windows для бюджетных компьютеров. Об этом сообщает издание Windows Latest. Журналисты выяснили, что операцион…
{'verb': 'выпустить', 'subject': 'microsoft', 'object': 'версия'}
{'verb': 'сообщать', 'subject': 'издание', 'object': None}
{'verb': 'выяснить', 'subject': 'журналист', 'object': None}
{'verb': 'иметь', 'subject': 'система', 'object': 'версия'}
{'verb': 'рассчитать', 'subject': None, 'object': None}
Topic: sports (doc 1)
Text: Британский боксер-тяжеловес Тайсон Фьюри оскорбил украинца Александра Усика. Его слова приводит Daily Mail. Спортсмен назвал возможного соперника маленьким укра…
{'verb': 'оскорбить', 'subject': 'боксёр', 'object': 'украинец'}
{'verb': 'приводить', 'subject': 'daily', 'object': 'слово'}
{'verb': 'назвать', 'subject': 'спортсмен', 'object': 'соперник'}
{'verb': 'заявить', 'subject': None, 'object': None}
{'verb': 'являться', 'subject': 'он', 'object': None}
Topic: economy (do

## 4. Частотные словосочетания

Найдите словосочетания по связям `amod` (прилагательное + существительное) и `nmod` (существительное + существительное). Для каждой пары сохраняйте сначала главное слово, затем зависимое. Учитывайте только буквенные токены (`is_alpha`), чтобы исключить числа и даты.

Сделайте `noun_phrases(doc, normalize=True)`. При `normalize=False` функция возвращает пары исходных словоформ (в нижнем регистре), при `True` - леммы или основы слов, полученные стеммингом. Подсчитайте десять наиболее частых пар по всем 12 статьям в обоих вариантах.

Проверьте, что, к примеру, формы "новая модель", "новой модели", "новые модели" после нормализации объединяются в одну пару. (как мини тест стемминга)

In [31]:
def noun_phrases(doc, normalize=True):
    pairs = []
    # Your code here
    NOUN_POS = {"NOUN", "PROPN"}

    for tok in doc:
        if not tok.is_alpha:
            continue

        if tok.dep_ == "amod":
            head, dep = tok.head, tok
            if head.pos_ not in NOUN_POS or tok.pos_ != "ADJ":
                continue
        elif tok.dep_ == "nmod":
            head, dep = tok.head, tok
            if head.pos_ not in NOUN_POS or tok.pos_ not in NOUN_POS:
                continue
        else:
            continue

        if not (head.is_alpha and dep.is_alpha):
            continue

        if normalize:
            h = head.lemma_.lower() or head.text.lower()
            d = dep.lemma_.lower()  or dep.text.lower()
            pairs.append((h, d))
        else:
            pairs.append((head.text.lower(), dep.text.lower()))

    return pairs

# Your code here
# +
# Проверьте три формы словосочетания "новая модель"

def count_pairs(docs_list, normalize):
    counter = Counter()
    for d in docs_list:
        for pair in noun_phrases(d, normalize=normalize):
            counter[pair] += 1
    return counter

raw_counter  = count_pairs(docs, normalize=False)
norm_counter = count_pairs(docs, normalize=True)

print("Top-10 pairs (source)")
for (h, d_), c in raw_counter.most_common(10):
    print(f"{h} {d_}: {c}")

print("\nTop-10 pairs (lemmas)")
for (h, d_), c in norm_counter.most_common(10):
    print(f"{h} {d_}: {c}")

Top-10 pairs (source)
интеллект искусственный: 5
система операционная: 3
весной года: 3
рынке российском: 3
интеллекта искусственного: 3
олимпиады национальной: 3
олимпиады технологической: 3
систему операционную: 2
организации боксерской: 2
рынок российский: 2

Top-10 pairs (lemmas)
интеллект искусственный: 8
система операционный: 6
рынок российский: 5
решение мегафон: 4
весна год: 3
олимпиада национальный: 3
олимпиада технологический: 3
распознавание текст: 3
обучение машинный: 3
потребление вода: 3


In [30]:
for text in ["новая модель", "новой модели", "новые модели"]:
    d = nlp(text)
    raw = noun_phrases(d, normalize=False)
    norm = noun_phrases(d, normalize=True)
    print(f"{text[:20]}  raw={raw}  norm={norm}")

новая модель  raw=[('модель', 'новая')]  norm=[('модель', 'новый')]
новой модели  raw=[('модели', 'новой')]  norm=[('модель', 'новый')]
новые модели  raw=[('модели', 'новые')]  norm=[('модель', 'новый')]


## 5. Выводы

Сравните две статьи разных тем. В 5–8 предложениях напишите:

1. Какие леммы чаще всего встречаются в статьях и насколько они отражают тему?
2. Какие сочетания "подлежащее - действие - дополнение" удалось найти? Что не получилось извлечь?
3. Какие словосочетания объединились после нормализации?
4. Приведите пример ошибки или ограничения предложенных правил.


In [ ]:
# Можно дописать тут код с примерами


**Выводы:**



## Дополнительное задание (необязательно): классификация новостей

Обучите классификатор тем новостей. Используйте частоты лемм и морфологические признаки. В корпусе уже есть метки и разделение на `train` и `test`. Много времени это занять не должно

Напишите `topic_features(doc)` - функцию, возвращающую словарь числовых признаков. Можно взять доли частей речи и частоты лемм. Следующая ячейка обучает `LogisticRegression` и показывает качество на тестовой части.

Чтобы попробовать больше классов, добавьте нужные номера в `TOPICS` в ячейке загрузки. Полный список: `0 climate`, `1 conflicts`, `2 culture`, `3 economy`, `4 gloss`, `5 health`, `6 politics`, `7 science`, `8 society`, `9 sports`, `10 travel`.

Чтобы изменить число статей каждого класса, поменяйте `COUNT_EXP` и снова выполните ячейку загрузки, а затем ячейки обработки. Корпус повторно скачивать не требуется, он сохраняется локально при первом запуске. Код классификатора менять не нужно.


In [ ]:
import math
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_df = texts_df[texts_df.source_split == 'train'].copy()
test_df = texts_df[texts_df.source_split == 'test'].copy()
print('Train:', train_df.topic.value_counts().to_dict())
print('Test:', test_df.topic.value_counts().to_dict())

#Your feature extractor
def topic_features(doc):
    # code here
    return {}

train_docs = list(nlp.pipe(train_df.text.tolist(), batch_size=16))
test_docs = list(nlp.pipe(test_df.text.tolist(), batch_size=16))
train_features = [topic_features(doc) for doc in train_docs]
test_features = [topic_features(doc) for doc in test_docs]

vectorizer = DictVectorizer()
X_train = vectorizer.fit_transform(train_features)
X_test = vectorizer.transform(test_features)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_df.topic)
predicted = model.predict(X_test)
print('Shape:', X_train.shape, X_test.shape)
print('Accuracy:', round(accuracy_score(test_df.topic, predicted), 4))
print(classification_report(test_df.topic, predicted, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(test_df.topic, predicted, labels=sorted(train_df.topic.unique())))

**Краткий вывод по дополнительному заданию:** какие признаки помогают отличать темы? Достаточно ли одних частей речи?
